In [22]:
from google.colab import files
uploaded = files.upload()

Saving fake_job_postings.csv to fake_job_postings (1).csv


In [29]:
!pip install pandas numpy scikit-learn nltk


In [23]:
import pandas as pd
import numpy as np

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [30]:
df =pd.read_csv("fake_job_postings.csv")
df.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   job_id               17880 non-null  int64 
 1   title                17880 non-null  object
 2   location             17534 non-null  object
 3   department           6333 non-null   object
 4   salary_range         2868 non-null   object
 5   company_profile      14572 non-null  object
 6   description          17879 non-null  object
 7   requirements         15184 non-null  object
 8   benefits             10668 non-null  object
 9   telecommuting        17880 non-null  int64 
 10  has_company_logo     17880 non-null  int64 
 11  has_questions        17880 non-null  int64 
 12  employment_type      14409 non-null  object
 13  required_experience  10830 non-null  object
 14  required_education   9775 non-null   object
 15  industry             12977 non-null  object
 16  func

In [34]:
df.columns

Index(['description', 'fraudulent'], dtype='object')

In [35]:
df = df[['description', 'fraudulent']]
df = df.dropna()
df.head()

,description,fraudulent
0,"Food52, a fast-growing, James Beard Award-winn...",0
1,Organised - Focused - Vibrant - Awesome!Do you...,0
2,"Our client, located in Houston, is actively se...",0
3,THE COMPANY: ESRI – Environmental Systems Rese...,0
4,JOB TITLE: Itemization Review ManagerLOCATION:...,0


In [36]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [37]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

stop_words = set(stopwords.words('english'))

def preprocess(text):
    words = word_tokenize(text.lower())        # convert to lowercase + tokenize
    words = [word for word in words if word.isalnum()]   # remove symbols
    words = [word for word in words if word not in stop_words]  # remove stopwords
    return " ".join(words)

df['cleaned'] = df['description'].apply(preprocess)

df[['description', 'cleaned']].head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,description,cleaned
0,"Food52, a fast-growing, James Beard Award-winn...",food52 james beard online food community curat...
1,Organised - Focused - Vibrant - Awesome!Do you...,organised focused vibrant awesome passion cust...
2,"Our client, located in Houston, is actively se...",client located houston actively seeking experi...
3,THE COMPANY: ESRI – Environmental Systems Rese...,company esri environmental systems research in...
4,JOB TITLE: Itemization Review ManagerLOCATION:...,job title itemization review managerlocation f...


In [38]:
df[['description', 'cleaned']].head()

,description,cleaned
0,"Food52, a fast-growing, James Beard Award-winn...",food52 james beard online food community curat...
1,Organised - Focused - Vibrant - Awesome!Do you...,organised focused vibrant awesome passion cust...
2,"Our client, located in Houston, is actively se...",client located houston actively seeking experi...
3,THE COMPANY: ESRI – Environmental Systems Rese...,company esri environmental systems research in...
4,JOB TITLE: Itemization Review ManagerLOCATION:...,job title itemization review managerlocation f...


In [39]:
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['cleaned'])
y = df['fraudulent']

In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [41]:
model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [42]:
y.value_counts()

,count
fraudulent,
0,17014
1,865


In [55]:
df['fraudulent'].value_counts()

,count
fraudulent,
0,17014
1,865


In [56]:
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [57]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9482662192393736
              precision    recall  f1-score   support

           0       0.99      0.96      0.97      3394
           1       0.49      0.77      0.60       182

    accuracy                           0.95      3576
   macro avg       0.74      0.87      0.79      3576
weighted avg       0.96      0.95      0.95      3576



In [58]:
def predict_job(text):
    text = preprocess(text)
    vector = vectorizer.transform([text])
    prediction = model.predict(vector)

    if prediction[0] == 1:
        return "Fake Job"
    else:
        return "Real Job"

In [59]:
predict_job("Earn money quickly without experience work from home")

'Fake Job'

In [60]:
predict_job("Earn $5000 weekly from home no experience required click now limited offer")

'Fake Job'

In [61]:
predict_job("We are hiring a software engineer with experience in Python and machine learning apply with your resume")

'Real Job'

In [62]:
predict_job("No skills needed instant hiring earn money fast apply now")

'Fake Job'

In [63]:
predict_job("Work from home opportunity flexible hours good communication skills required")

'Fake Job'

In [64]:
predict_job("Earn 10000 dollars per week from home no experience needed limited slots apply now")

'Fake Job'

In [65]:
predict_job("Instant hiring work from home job no skills required start earning today")

'Fake Job'

In [66]:
import pickle

# Save model
pickle.dump(model, open('fake_job_model.pkl', 'wb'))

# Save vectorizer
pickle.dump(vectorizer, open('vectorizer.pkl', 'wb'))

In [67]:
print("Fake Job Posting Detection Model Built Successfully!")

Fake Job Posting Detection Model Built Successfully!
